In [1]:
# ============================================================
# Gold Layer - Business Aggregation
# Fleet Data Platform Modernisation
# ============================================================

storage_account = "stfleetcndatalakeprod"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/fleet_telematics/"
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/fact_fleet_performance/"

print(f"Reading from: {silver_path}")
print(f"Writing to: {gold_path}")

StatementMeta(sparkpool01, 7, 2, Finished, Available, Finished, False)

Reading from: abfss://silver@stfleetcndatalakeprod.dfs.core.windows.net/fleet_telematics/
Writing to: abfss://gold@stfleetcndatalakeprod.dfs.core.windows.net/fact_fleet_performance/


In [2]:
from pyspark.sql import functions as F

df_silver = spark.read.format("delta").load(silver_path)

print(f"Records from Silver: {df_silver.count()}")
df_silver.printSchema()

StatementMeta(sparkpool01, 7, 3, Finished, Available, Finished, False)

Records from Silver: 100
root
 |-- event_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- fuel_level_pct: double (nullable = true)
 |-- engine_temperature_c: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- is_overspeeding: boolean (nullable = true)
 |-- ingestion_audit_timestamp: timestamp (nullable = true)



In [3]:
df_gold = df_silver.groupBy("vehicle_id").agg(
    F.max("speed_mph").alias("maximum_logged_speed"),
    F.round(F.avg("speed_mph"), 2).alias("average_operational_speed"),
    F.round(F.avg("fuel_level_pct"), 2).alias("mean_fuel_capacity"),
    F.sum(F.col("is_overspeeding").cast("int")).alias("total_overspeeding_violations"),
    F.count("event_id").alias("total_events_recorded")
).withColumn(
    "last_modified_date", F.current_timestamp()
)

print(f"Vehicles aggregated: {df_gold.count()}")
df_gold.show()

StatementMeta(sparkpool01, 7, 4, Finished, Available, Finished, False)

Vehicles aggregated: 10
+----------+--------------------+-------------------------+------------------+-----------------------------+---------------------+--------------------+
|vehicle_id|maximum_logged_speed|average_operational_speed|mean_fuel_capacity|total_overspeeding_violations|total_events_recorded|  last_modified_date|
+----------+--------------------+-------------------------+------------------+-----------------------------+---------------------+--------------------+
|    VH-001|               79.71|                    38.51|             50.52|                            3|                   10|2026-06-17 08:12:...|
|    VH-010|               63.56|                    26.96|             52.71|                            1|                   14|2026-06-17 08:12:...|
|    VH-005|               66.84|                    44.32|             52.09|                            2|                   10|2026-06-17 08:12:...|
|    VH-009|               55.47|                     38.1|     

In [4]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("Gold Delta written successfully")
df_gold.orderBy(F.desc("total_overspeeding_violations")).show()

StatementMeta(sparkpool01, 7, 5, Finished, Available, Finished, False)

Gold Delta written successfully
+----------+--------------------+-------------------------+------------------+-----------------------------+---------------------+--------------------+
|vehicle_id|maximum_logged_speed|average_operational_speed|mean_fuel_capacity|total_overspeeding_violations|total_events_recorded|  last_modified_date|
+----------+--------------------+-------------------------+------------------+-----------------------------+---------------------+--------------------+
|    VH-002|               74.23|                     51.3|             56.58|                            7|                   15|2026-06-17 08:12:...|
|    VH-008|               84.56|                    42.72|             51.07|                            4|                   13|2026-06-17 08:12:...|
|    VH-004|               75.88|                     51.0|             60.46|                            4|                    8|2026-06-17 08:12:...|
|    VH-001|               79.71|                    38.